### Loading the triage_200 csv datafile



In [4]:
import pandas as pd
df=pd.read_csv(r"C:\Users\Deepalakshmi\Downloads\sample_emails_with_triage_200.csv")

In [5]:
#select 100 test emails
eval_df = df.sample(100,random_state=42)
eval_df = eval_df.reset_index(drop=True)
eval_df.head()

,id,sender,subject,body,priority,triage_label
0,96,no-reply@service.com,Weekly Newsletter,Your order #3634 has been shipped and is expec...,low,ignore
1,16,news@techblog.com,Payment Overdue,"Hi, don't miss our sale with discounts up to 7...",low,notify_human
2,31,news@techblog.com,Weekly Newsletter,Notice: Your account will be locked unless ver...,high,notify_human
3,159,no-reply@service.com,Welcome to Service,Reminder: The client meeting is scheduled at 9...,low,respond
4,129,sales@shop.com,Invoice Due,Your order #6464 has been shipped and is expec...,low,respond


### Creating Ideal_response column

In [6]:
columns = ["id","sender","subject","body","priority","triage_label"]
df = pd.DataFrame(df, columns=columns)

# Function using if-elif-else
def ideal_response(label):
    if label == "notify_human":
        return "Escalate to human support"
    elif label == "respond":
        return "Send automated response"
    elif label == "ignore":
        return "No action required"
    else:
        return "Manual review needed"

# Create new column
df["ideal_response"] = df["triage_label"].apply(ideal_response)

df[["triage_label","ideal_response"]].head()


,triage_label,ideal_response
0,notify_human,Escalate to human support
1,respond,Send automated response
2,ignore,No action required
3,respond,Send automated response
4,respond,Send automated response


# Milestone2_Lingaeswari

### Loading Evaluation_sample dataset


In [7]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Deepalakshmi\Downloads\email_evaluation_dataset_Lingaeswari.csv")

In [9]:
df.head()

,id,email_text,expected_action,expected_tone,clean_text
0,1,This is a reminder to attend the client discus...,notify,neutral,this is a reminder to attend the client discus...
1,2,Please remember the project review meeting hap...,notify,neutral,please remember the project review meeting hap...
2,3,This is a reminder to attend the client discus...,notify,neutral,this is a reminder to attend the client discus...
3,4,This is a reminder to attend the weekly update...,notify,neutral,this is a reminder to attend the weekly update...
4,5,This is a reminder to attend the team sync mee...,notify,neutral,this is a reminder to attend the team sync mee...


### Cleaning email text

In [8]:
df['clean_text'] = (
    df['email_text']
    .str.lower()
    .str.replace('[^a-zA-Z ]', '', regex=True)
)

df[['email_text', 'clean_text']].head()

,email_text,clean_text
0,This is a reminder to attend the client discus...,this is a reminder to attend the client discus...
1,Please remember the project review meeting hap...,please remember the project review meeting hap...
2,This is a reminder to attend the client discus...,this is a reminder to attend the client discus...
3,This is a reminder to attend the weekly update...,this is a reminder to attend the weekly update...
4,This is a reminder to attend the team sync mee...,this is a reminder to attend the team sync mee...


### Email assistant function

In [ ]:
def email_assistant(email_text):
    text = email_text.lower()
    
    if "urgent" in text or "submit" in text or "deadline" in text:
        return "notify", "urgent"
    
    elif "thank you" in text:
        return "ignore", "polite"
    
    else:
        return "review", "neutral"

### 1. Applying assistant function to the dataset

In [11]:
# Apply assistant logic to each email
df["prediction"] = df["email_text"].apply(email_assistant)
df["prediction"]


0     (review, neutral)
1     (review, neutral)
2     (review, neutral)
3     (review, neutral)
4     (review, neutral)
            ...        
95     (ignore, polite)
96    (review, neutral)
97     (ignore, polite)
98     (ignore, polite)
99     (ignore, polite)
Name: prediction, Length: 100, dtype: object

### 2. Creating separate output columns

In [13]:
df[["predicted_action", "predicted_tone"]] = pd.DataFrame(
    df["prediction"].tolist(), index=df.index
)


In [14]:
df[["predicted_action", "predicted_tone"]] 

,predicted_action,predicted_tone
0,review,neutral
1,review,neutral
2,review,neutral
3,review,neutral
4,review,neutral
...,...,...
95,ignore,polite
96,review,neutral
97,ignore,polite
98,ignore,polite


### 3. Compare predictions with expected values

In [15]:
df["action_correct"] = df["predicted_action"] == df["expected_action"]
df["tone_correct"] = df["predicted_tone"] == df["expected_tone"]
df.head()

,id,email_text,expected_action,expected_tone,clean_text,prediction,predicted_action,predicted_tone,action_correct,tone_correct
0,1,This is a reminder to attend the client discus...,notify,neutral,this is a reminder to attend the client discus...,"(review, neutral)",review,neutral,False,True
1,2,Please remember the project review meeting hap...,notify,neutral,please remember the project review meeting hap...,"(review, neutral)",review,neutral,False,True
2,3,This is a reminder to attend the client discus...,notify,neutral,this is a reminder to attend the client discus...,"(review, neutral)",review,neutral,False,True
3,4,This is a reminder to attend the weekly update...,notify,neutral,this is a reminder to attend the weekly update...,"(review, neutral)",review,neutral,False,True
4,5,This is a reminder to attend the team sync mee...,notify,neutral,this is a reminder to attend the team sync mee...,"(review, neutral)",review,neutral,False,True


### 4. Calculating accuracy

In [16]:
action_accuracy = df["action_correct"].mean() * 100
tone_accuracy = df["tone_correct"].mean() * 100

print(f"Action Accuracy: {action_accuracy:.2f}%")
print(f"Tone Accuracy: {tone_accuracy:.2f}%")


Action Accuracy: 10.00%
Tone Accuracy: 30.00%


### 5. Perform error analysis

#### Action errors

In [17]:
action_errors = df[df["action_correct"] == False][
    ["email_text", "expected_action", "predicted_action"]
]
action_errors


,email_text,expected_action,predicted_action
0,This is a reminder to attend the client discus...,notify,review
1,Please remember the project review meeting hap...,notify,review
2,This is a reminder to attend the client discus...,notify,review
3,This is a reminder to attend the weekly update...,notify,review
4,This is a reminder to attend the team sync mee...,notify,review
...,...,...,...
90,This email is to inform you about recent updat...,ignore,review
91,This email is to inform you about recent updat...,ignore,review
92,This email is to inform you about recent updat...,ignore,review
94,This email is to inform you about recent updat...,ignore,review


#### Tone errors

In [18]:
tone_errors = df[df["tone_correct"] == False][
    ["email_text", "expected_tone", "predicted_tone"]
]
tone_errors


,email_text,expected_tone,predicted_tone
10,Payment reminder: Your invoice INV-2000 amount...,urgent,neutral
11,"Dear Customer, invoice INV-2001 for INR 6500 i...",urgent,neutral
12,Payment reminder: Your invoice INV-2002 amount...,urgent,neutral
13,"Dear Customer, invoice INV-2003 for INR 6500 i...",urgent,neutral
14,"Dear Customer, invoice INV-2004 for INR 6500 i...",urgent,neutral
...,...,...,...
93,Thank you for participating in the event. This...,neutral,polite
95,Thank you for participating in the event. This...,neutral,polite
97,Thank you for participating in the event. This...,neutral,polite
98,Thank you for participating in the event. This...,neutral,polite


### 6. Save your output CSV

In [19]:
df.to_csv(r"C:\Users\Deepalakshmi\Downloads\data\milestone2_output_lingaeswari.csv", index=False)


In [20]:
pip install langsmith

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
from langsmith import Client
client= Client()

In [22]:
#Define the Judge Prompt

judge_prompt = """You are an evaluator.Compare the model output with the ideal answer.

Check:
1. Action correctness
2. Tone correctness

Give score:
1 = correct
0 = incorrect
"""

In [60]:
#Run agent + Judge
def evaluate(agent_output, ideal_action, ideal_tone):
    if (
        agent_output["action"]== ideal_action 
        and agent_output["tone"] == ideal_tone
    ):
        return 1
    else:
        return 0

In [61]:
agent_output ={
    "action":"notify",
    "tone":"urgent"
}

In [62]:
ideal_action = "notify"
ideal_tone = "urgent"

In [63]:
score = evaluate(agent_output, ideal_action, ideal_tone)
score

1

# --------------------------------------------------------------

# Milestone 2 – Evaluation

### PART 1: Dataset Preparation

In [2]:
import pandas as pd
df=pd.read_csv(r"C:\Users\Deepalakshmi\Downloads\sample_emails_with_triage_200.csv")

In [4]:
def add_ground_truth(body):
    text = str(body).lower()

    if any(w in text for w in ["submit", "deadline", "meeting", "reminder", "eod", "asap"]):
        return "notify", "urgent"
    elif any(w in text for w in ["thank you", "thanks"]):
        return "ignore", "polite"
    elif any(w in text for w in ["please", "can you", "could you", "review", "confirm"]):
        return "respond", "polite"
    else:
        return "", "neutral"


df[["ideal_intent", "ideal_tone"]] = df["body"].apply(
    lambda x: pd.Series(add_ground_truth(x))
)    

In [5]:
df[["body", "ideal_intent", "ideal_tone"]].head()

,body,ideal_intent,ideal_tone
0,Reminder: The client meeting is scheduled at 1...,notify,urgent
1,Your invoice of INR 25515.09 is due on 2025-12...,respond,polite
2,Reminder: The client meeting is scheduled at 1...,notify,urgent
3,"Hello team, please find the attached weekly re...",respond,polite
4,"Hello team, please find the attached weekly re...",respond,polite


### PART 2: Evaluation Notebook

In [15]:
#Created milestone2_evaluation_lingaeswari.ipynb notebook file.

In [9]:
import pandas as pd
df.to_csv(r"C:\Users\Deepalakshmi\Downloads\sample_emails_with_triage_200.csv",index="False")
df.head()

,Unnamed: 0,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,notify,urgent
1,1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,polite
2,2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,notify,urgent
3,3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,polite
4,4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,polite


In [32]:
df.columns

Index(['id', 'sender', 'subject', 'body', 'priority', 'triage_label',
       'ideal_intent', 'ideal_tone'],
      dtype='object')

### PART 3: Email Assistant Logic


In [10]:
def email_assistant(email_text):
    text = str(email_text).lower()

    # URGENT / NOTIFY
    if any(word in text for word in [
        "urgent", "asap", "immediately", "submit", "deadline", "eod", "today"
    ]):
        return "notify", "urgent"

    # RESPOND / POLITE
    elif any(word in text for word in [
        "please", "can you", "could you", "kindly", "review", "confirm", "reply"
    ]):
        return "respond", "polite"

    # IGNORE / POLITE
    elif any(word in text for word in [
        "thank you", "thanks", "appreciate", "grateful"
    ]):
        return "ignore", "polite"

    # INFORMATIONAL / NEUTRAL
    elif any(word in text for word in [
        "meeting", "schedule", "reminder", "update", "announcement"
    ]):
        return "notify", "neutral"

    # DEFAULT (UNCLEAR)
    else:
        return "review", "neutral"



### PART 4: Generate Predictions


In [11]:
predictions = []
for _, row in df.iterrows():
 action, tone = email_assistant(row["body"])
 predictions.append({
 "id": row["id"],
 "predicted_intent": action,
 "predicted_tone": tone
 })
pred_df = pd.DataFrame(predictions)
pred_df.head()


,id,predicted_intent,predicted_tone
0,1,notify,neutral
1,2,respond,polite
2,3,notify,neutral
3,4,respond,polite
4,5,respond,polite


### PART 5: Evaluate Accuracy

In [12]:
def evaluate(row):
    score = 0
    if row["predicted_intent"] == row["ideal_intent"]:
        score += 1
    if row["predicted_tone"] == row["ideal_tone"]:
        score += 1
    return score

eval_df = df.merge(pred_df, on="id")
eval_df["score"] = eval_df.apply(evaluate, axis=1)
accuracy = (eval_df["score"].sum() / (len(eval_df) * 2)) * 100
accuracy

57.99999999999999

### PART 6: Save Evaluation Output

In [13]:
output_path = r"C:\Users\Deepalakshmi\Downloads\data\milestone2_output_lingaeswari.csv"

eval_df.to_csv(output_path, index=False)

print("Saved to:", output_path)


Saved to: C:\Users\Deepalakshmi\Downloads\data\milestone2_output_lingaeswari.csv
